# NBA 3.1 — v4 Indication × Drug-Pool Decile Analysis

For each of **4 indication groups** (T2D, CKD, HF, T1D), compute **3 drug-pool deciles**:

| Drug Pool | Product Classes |
|-----------|----------------|
| **SGLT2/GLP1** | SGLT2 + GLP1 (pooled) |
| **ACE/ARB** | ACE + ARB (pooled) |
| **sMRA** | sMRA alone |

→ **12 decile sets** total (4 indications × 3 pools)

**Decile method:** Cumulative volume-based (each decile ≈ 10% of total volume)  
**Source:** `lrx_nbrx` — `INDICATION_GROUP_APP1` for claim-level indication × `PRODUCT_CLASS` for drug pool  
**Metric:** `sum(NBRX)` per PROVIDER_ID within each indication × pool

In [0]:
%run /Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA\ 3.0\ Ops\ -\ Pipeline/00_Helper_Notebooks/00_connection

In [0]:
%run /Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA\ 3.0\ Ops\ -\ Pipeline/00_Helper_Notebooks/00_config

In [0]:
from pyspark.sql import functions as F, Window
import builtins
import itertools

CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
SCHEMA  = "pace_ker_in_house_schema"
LOOKBACK_START = "2024-10-01"
LOOKBACK_END   = "2026-09-30"
DECILE_CUTOFF  = 2

# 4 indication groups (from INDICATION_GROUP_APP1)
INDICATION_GROUPS = ["T2D", "CKD", "HF", "T1D"]

# 3 drug pools (keys = pool name, values = list of PRODUCT_CLASS values to combine)
DRUG_POOLS = {
    "SGLT2_GLP1": ["SGLT2", "GLP1"],
    "ACE_ARB":    ["ACE", "ARB"],
    "sMRA":       ["sMRA"],
}

print(f"Lookback       : {LOOKBACK_START} → {LOOKBACK_END}")
print(f"Indications    : {INDICATION_GROUPS}")
print(f"Drug pools     : {list(DRUG_POOLS.keys())}")
print(f"Decile cutoff  : {DECILE_CUTOFF}+")
print(f"Combinations   : {len(INDICATION_GROUPS) * len(DRUG_POOLS)} decile sets")

In [0]:
# Load lrx_nbrx (has INDICATION_GROUP_APP1 for claim-level indication mapping)
lrx_nbrx = spark.table(f"`{CATALOG}`.`{SCHEMA}`.lrx_nbrx")

lrx_window = lrx_nbrx.filter(
    (F.col("WEEK_ENDING_DATE") >= LOOKBACK_START) &
    (F.col("WEEK_ENDING_DATE") <= LOOKBACK_END)
)

# Check INDICATION_GROUP_APP1 values
print("=== INDICATION_GROUP_APP1 distinct values ===")
lrx_window.select("INDICATION_GROUP_APP1").distinct().orderBy("INDICATION_GROUP_APP1").show(20, False)

# Check PRODUCT_CLASS values
print("\n=== PRODUCT_CLASS distinct values ===")
lrx_window.select("PRODUCT_CLASS").distinct().orderBy("PRODUCT_CLASS").show(20, False)

print(f"\nlrx_window total rows: {lrx_window.count():,}")

In [0]:
# Check if pt_lvl_nbrx has INDICATION_GROUP_APP1 at claim level
# (would allow count-based TRx-equivalent deciling with indication)
pt_lvl_nbrx = spark.table(f"`{CATALOG}`.`{SCHEMA}`.pt_lvl_nbrx")
print("pt_lvl_nbrx columns:")
print(pt_lvl_nbrx.columns)
print(f"\nHas INDICATION_GROUP_APP1: {'INDICATION_GROUP_APP1' in pt_lvl_nbrx.columns}")
print(f"Has PRODUCT_CLASS: {'PRODUCT_CLASS' in pt_lvl_nbrx.columns}")

In [0]:
def compute_cumulative_decile(df, indication, drug_pool_name, drug_pool_classes, cutoff=DECILE_CUTOFF):
    """
    Compute cumulative NBRX-based deciles for a given indication × drug pool.
    Returns (all_deciles_df, d2plus_df, stats_dict).
    """
    # Filter to indication AND drug pool product classes
    filtered = df.filter(
        (F.col("INDICATION_GROUP_APP1").contains(indication)) &
        (F.col("PRODUCT_CLASS").isin(drug_pool_classes))
    )
    
    # Sum NBRX per PROVIDER_ID
    hcp_vol = filtered.groupBy("PROVIDER_ID").agg(F.sum("NBRX").alias("TOTAL_NBRX"))
    
    total_hcps = hcp_vol.count()
    if total_hcps == 0:
        return None, None, {"indication": indication, "drug_pool": drug_pool_name,
                            "total_hcps": 0, "d2plus": 0, "d1_dropped": 0}
    
    # Cumulative volume-based decile (matching v1 methodology)
    total_vol = hcp_vol.agg(F.sum("TOTAL_NBRX")).collect()[0][0]
    vol_per_decile = total_vol / 10
    
    # PROVIDER_ID tiebreaker ensures row-by-row cumsum (matching v1 pandas behavior)
    w = Window.orderBy(F.asc("TOTAL_NBRX"), F.asc("PROVIDER_ID"))
    hcp_vol = hcp_vol.withColumn("CUM_NBRX", F.sum("TOTAL_NBRX").over(w))
    hcp_vol = hcp_vol.withColumn(
        "DECILE",
        F.least(
            (F.col("CUM_NBRX") / F.lit(vol_per_decile)).cast("int") + 1,
            F.lit(10)
        )
    ).drop("CUM_NBRX")
    
    d2plus = hcp_vol.filter(F.col("DECILE") >= cutoff)
    d2_count = d2plus.count()
    
    stats = {
        "indication": indication,
        "drug_pool": drug_pool_name,
        "total_hcps": total_hcps,
        "total_nbrx": int(total_vol),
        "d2plus": d2_count,
        "d1_dropped": total_hcps - d2_count,
        "d1_pct": round(100 * (total_hcps - d2_count) / total_hcps, 1),
    }
    return hcp_vol, d2plus, stats

print("✓ Decile function defined")

In [0]:
# Run all 12 combinations: 4 indications × 3 drug pools
all_results = {}   # (indication, pool) -> {all_deciles, d2plus, stats}
all_stats = []

for indication in INDICATION_GROUPS:
    for pool_name, pool_classes in DRUG_POOLS.items():
        key = (indication, pool_name)
        all_dec, d2plus, stats = compute_cumulative_decile(
            lrx_window, indication, pool_name, pool_classes
        )
        all_results[key] = {"all_deciles": all_dec, "d2plus": d2plus, "stats": stats}
        all_stats.append(stats)
        
        print(f"{indication:<4} × {pool_name:<12}  "
              f"Total HCPs: {stats['total_hcps']:>10,}  |  "
              f"D2+: {stats['d2plus']:>10,}  |  "
              f"D1 dropped: {stats['d1_dropped']:>10,} ({stats.get('d1_pct', 0)}%)")
    print()  # blank line between indications

In [0]:
import pandas as pd

# Build summary DataFrame
df_summary = pd.DataFrame(all_stats)
df_summary = df_summary[["indication", "drug_pool", "total_hcps", "total_nbrx", "d2plus", "d1_dropped", "d1_pct"]]
df_summary.columns = ["Indication", "Drug Pool", "Total HCPs", "Total NBRX", "Decile 2+", "D1 Dropped", "D1 %"]

# Pivot: rows = indication, columns = drug pool, values = D2+ count
pivot_d2 = df_summary.pivot(index="Indication", columns="Drug Pool", values="Decile 2+")
pivot_d2 = pivot_d2.reindex(index=INDICATION_GROUPS, columns=list(DRUG_POOLS.keys()))

print("=== Decile 2+ HCP Counts (Indication × Drug Pool) ===")
print(pivot_d2.to_string())
print()

# Also pivot total HCPs
pivot_total = df_summary.pivot(index="Indication", columns="Drug Pool", values="Total HCPs")
pivot_total = pivot_total.reindex(index=INDICATION_GROUPS, columns=list(DRUG_POOLS.keys()))

print("=== Total HCPs (Indication × Drug Pool) ===")
print(pivot_total.to_string())
print()

# Full detail table
print("=== Full Detail ===")
df_summary.to_string(index=False)
display(spark.createDataFrame(df_summary))

In [0]:
# Show decile distribution for each of the 12 combinations
for indication in INDICATION_GROUPS:
    for pool_name in DRUG_POOLS.keys():
        key = (indication, pool_name)
        result = all_results[key]
        if result["all_deciles"] is None:
            print(f"{indication} × {pool_name}: NO DATA")
            continue
        
        print(f"\n--- {indication} × {pool_name} ---")
        result["all_deciles"].groupBy("DECILE").agg(
            F.count("*").alias("n_hcps"),
            F.sum("TOTAL_NBRX").alias("total_nbrx"),
        ).orderBy("DECILE").show()

In [0]:
# For each drug pool, compute pairwise overlap of D2+ HCPs across indications
print("=== Cross-Indication Overlap (D2+ HCPs) Per Drug Pool ===")

for pool_name in DRUG_POOLS.keys():
    print(f"\n--- Drug Pool: {pool_name} ---")
    print(f"{'Pair':<12} {'Overlap':>10} {'% of smaller':>14}")
    print("-" * 40)
    
    for a, b in itertools.combinations(INDICATION_GROUPS, 2):
        d2_a = all_results[(a, pool_name)]["d2plus"]
        d2_b = all_results[(b, pool_name)]["d2plus"]
        
        if d2_a is None or d2_b is None:
            print(f"{a}∩{b:<8} {'N/A':>10}")
            continue
        
        ids_a = d2_a.select("PROVIDER_ID").distinct()
        ids_b = d2_b.select("PROVIDER_ID").distinct()
        overlap = ids_a.join(ids_b, on="PROVIDER_ID", how="inner").count()
        
        count_a = all_results[(a, pool_name)]["stats"]["d2plus"]
        count_b = all_results[(b, pool_name)]["stats"]["d2plus"]
        smaller = builtins.min(count_a, count_b)
        pct = round(100 * overlap / smaller, 1) if smaller > 0 else 0
        
        print(f"{a}∩{b:<8} {overlap:>10,} {pct:>13.1f}%")

In [0]:
# For each indication, compute pairwise overlap of D2+ HCPs across drug pools
print("=== Cross-Drug-Pool Overlap (D2+ HCPs) Per Indication ===")

pool_names = list(DRUG_POOLS.keys())
for indication in INDICATION_GROUPS:
    print(f"\n--- Indication: {indication} ---")
    print(f"{'Pair':<25} {'Overlap':>10} {'% of smaller':>14}")
    print("-" * 52)
    
    for a, b in itertools.combinations(pool_names, 2):
        d2_a = all_results[(indication, a)]["d2plus"]
        d2_b = all_results[(indication, b)]["d2plus"]
        
        if d2_a is None or d2_b is None:
            print(f"{a}∩{b:<16} {'N/A':>10}")
            continue
        
        ids_a = d2_a.select("PROVIDER_ID").distinct()
        ids_b = d2_b.select("PROVIDER_ID").distinct()
        overlap = ids_a.join(ids_b, on="PROVIDER_ID", how="inner").count()
        
        count_a = all_results[(indication, a)]["stats"]["d2plus"]
        count_b = all_results[(indication, b)]["stats"]["d2plus"]
        smaller = builtins.min(count_a, count_b)
        pct = round(100 * overlap / smaller, 1) if smaller > 0 else 0
        
        print(f"{a} ∩ {b:<16} {overlap:>10,} {pct:>13.1f}%")

In [0]:
# Union and intersection across all 12 D2+ sets
from functools import reduce

# Collect all D2+ PROVIDER_ID sets
d2_sets = []
for indication in INDICATION_GROUPS:
    for pool_name in DRUG_POOLS.keys():
        d2 = all_results[(indication, pool_name)]["d2plus"]
        if d2 is not None:
            d2_sets.append(d2.select("PROVIDER_ID").distinct())

# Union of all 12
union_all = reduce(lambda a, b: a.unionByName(b), d2_sets).distinct()
union_count = union_all.count()

# Intersection of all 12
intersect_all = reduce(lambda a, b: a.join(b, on="PROVIDER_ID", how="inner"), d2_sets)
intersect_count = intersect_all.count()

print(f"Union of all 12 D2+ sets     : {union_count:,}")
print(f"Intersection of all 12 sets  : {intersect_count:,}")
print(f"Intersection / Union ratio   : {100*intersect_count/union_count:.1f}%")

# Per-indication union (any of 3 pools)
print("\n=== Per-Indication Union (any drug pool D2+) ===")
for indication in INDICATION_GROUPS:
    pools = [all_results[(indication, p)]["d2plus"].select("PROVIDER_ID").distinct()
             for p in DRUG_POOLS.keys()
             if all_results[(indication, p)]["d2plus"] is not None]
    if pools:
        ind_union = reduce(lambda a, b: a.unionByName(b), pools).distinct()
        print(f"{indication}: {ind_union.count():,}")

# Per-pool union (any of 4 indications)
print("\n=== Per-Drug-Pool Union (any indication D2+) ===")
for pool_name in DRUG_POOLS.keys():
    inds = [all_results[(i, pool_name)]["d2plus"].select("PROVIDER_ID").distinct()
            for i in INDICATION_GROUPS
            if all_results[(i, pool_name)]["d2plus"] is not None]
    if inds:
        pool_union = reduce(lambda a, b: a.unionByName(b), inds).distinct()
        print(f"{pool_name}: {pool_union.count():,}")